# Wild-Diff-ICMH — Phase 2 trên Google Colab

Notebook này dựng hạ tầng đánh giá và các hàng đối chứng, **không train**. Kết quả nằm trong
`MyDrive/wild_diff_icmh/phase2/` và `results/results.jsonl`.

| Bước | Việc | GPU? |
|---|---|---|
| 1–5 | Drive, code (nhánh `phase2`), cài đặt, chép ảnh, checkpoint (giống notebook training) | có runtime GPU |
| P2-0 | Checkpoint tác giả λ = 2/8/32 (tải từ Hugging Face về Drive một lần) | không |
| P2-1 | Nhãn ngày/đêm theo nguồn sáng cho 10.222 ảnh | không |
| P2-2 | Đóng băng tập dev 300 ảnh | không |
| P2-3 | MegaDetector trên ảnh gốc (env riêng) | có |
| P2-4 | Thống kê miền | không |
| P2-5 | RAM++ tags cho cả 20 site | có |
| P2-6 | Đo thời gian decode B0 ở 1024 và 512 → dự báo CU | có |
| P2-7 | **Decode B0 trên tập dev** — chỉ chạy khi đặt `RUN_B0 = True` | có, tốn nhất |
| P2-8 | Baseline JPEG/WebP | không |
| P2-9 | Baseline CompressAI | có, nhẹ |
| P2-10 | Chấm điểm mọi kho lưu trữ (ảnh + MegaDetector) | có |
| P2-11 | Đồ thị RD chung | không |
| P2-12 | Ghi CU đo thật của phiên | không |

Mọi bước đều **chạy tiếp được**: runtime mới thì chạy lại 1–5 rồi chạy lại các bước P2; phần đã xong
tự bỏ qua. Ngân sách Phase 2: ≤ 8 CU — xem dự báo ở P2-6 trước khi bật P2-7.

## Bước 1 — Gắn Google Drive (mỗi runtime mới)

Trước khi chạy, mở **Runtime → View resources** và chép số compute unit
**"Available"** vào `CU_AVAILABLE_AT_START`. Bước 10 dùng số này để tính CU thật đã tiêu.

In [ ]:
CU_AVAILABLE_AT_START = None  # ví dụ 74.37 — số "Available" lúc bắt đầu phiên

from google.colab import drive

drive.mount('/content/drive')
if CU_AVAILABLE_AT_START is None:
    print('Nhắc: chưa điền CU_AVAILABLE_AT_START; Bước 10 sẽ yêu cầu số này.')


## Bước 2 — Lấy mã nguồn và kiểm tra GPU (mỗi runtime mới)

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/TranDuon/Wild-Diff-ICMH.git'
BRANCH = 'phase2'  # Phase 2 branch; set back to 'main' when merging
REPO = Path('/content/Wild-Diff-ICMH')

if (REPO / '.git').is_dir():
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO)], check=True)

os.chdir(REPO)
subprocess.run(['nvidia-smi'], check=True)
print('Mã nguồn:', REPO)


### Bước 2A — Cập nhật bản sửa mới nhất (chỉ chạy khi được báo đã push)

Cell cố định này thay cho mọi cell pull tạm. Trong cùng runtime, chạy cell này sau
khi có bản sửa trên GitHub, rồi chạy lại **đúng cell vừa lỗi**. Cell tự in commit
trước/sau và xác nhận thư mục code đã khớp hoàn toàn với `origin/main`.

In [ ]:
assert (REPO / '.git').is_dir(), 'Hãy chạy Bước 2 trước để clone repository.'

local_changes = subprocess.check_output(
    ['git', 'status', '--porcelain'], cwd=REPO, text=True
).strip()
if local_changes:
    raise RuntimeError(
        'Thư mục code Colab có sửa cục bộ nên chưa thể pull an toàn:\n'
        f'{local_changes}\n'
        'Không sửa file trong /content/Wild-Diff-ICMH; hãy gửi phần này để xử lý.'
    )

before = subprocess.check_output(
    ['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO, text=True
).strip()
subprocess.run(
    ['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO, check=True
)
after = subprocess.check_output(
    ['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO, text=True
).strip()
remote = subprocess.check_output(
    ['git', 'rev-parse', '--short', f'origin/{BRANCH}'], cwd=REPO, text=True
).strip()

print('Commit trước:', before)
print('Commit sau:  ', after)
print(f'GitHub {BRANCH}:', remote)
assert after == remote, f'Code local {after} chưa khớp GitHub {remote}'
if before == after:
    print('ĐÃ Ở BẢN MỚI NHẤT — chạy lại đúng cell vừa lỗi.')
else:
    print('CẬP NHẬT THÀNH CÔNG — chạy lại đúng cell vừa lỗi.')


## Bước 3 — Cài dependencies an toàn (mỗi runtime mới)

Cell này chủ động giữ nguyên NumPy/SciPy/Torch có sẵn của Colab,
cài đủ dependency runtime của CompressAI, rồi import `TagGCM` để bắt lỗi ngay tại đây.

In [ ]:
source_requirements = REPO / 'requirements-colab.txt'
safe_requirements = Path('/content/requirements-colab-safe.txt')
core_constraints = Path('/content/colab-core-constraints.txt')

from importlib import metadata as importlib_metadata

core_distributions = ('numpy', 'scipy', 'torch', 'torchvision')
core_versions_before = {
    name: importlib_metadata.version(name)
    for name in core_distributions
}

filtered_lines = []
for line in source_requirements.read_text(encoding='utf-8').splitlines():
    normalized = line.strip().lower()
    if normalized.startswith(('numpy', 'scipy')):
        print('Giữ bản Colab, bỏ qua:', line)
        continue
    filtered_lines.append(line)

safe_requirements.write_text(
    '\n'.join(filtered_lines) + '\n',
    encoding='utf-8',
)

core_constraints.write_text(
    '\n'.join(
        f'{name}=={version}'
        for name, version in core_versions_before.items()
    ) + '\n',
    encoding='utf-8',
)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', str(safe_requirements),
    '-c', str(core_constraints),
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--no-deps', '--no-build-isolation',
    'compressai==1.2.8',
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    '--force-reinstall', 'src/recognize-anything',
], cwd=REPO, check=True)

import importlib
import numpy as np
import numpy.random as npr
import scipy
import torch
import torch_geometric
import pytorch_msssim
import compressai.ans
import compressai.entropy_models
import compressai.losses
import compressai.zoo

# A pip subprocess can add files after this Jupyter kernel initialized.
# Refresh import caches so the freshly installed vendored package is
# visible immediately, without requiring a runtime restart.
importlib.invalidate_caches()
import ram
from model.lfgcm import TagGCM

core_versions_after = {
    name: importlib_metadata.version(name)
    for name in core_distributions
}
assert core_versions_after == core_versions_before, (
    'Dependency install changed Colab core packages: '
    f'before={core_versions_before}, after={core_versions_after}'
)
assert torch.cuda.is_available(), 'Vào Runtime > Change runtime type > chọn GPU'
gpu = torch.cuda.get_device_properties(0)
print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__, np.__file__)
print('SciPy:', scipy.__version__)
print('Torch:', torch.__version__, torch.version.cuda)
print('PyG:', torch_geometric.__version__)
print('RAM package:', ram.__file__)
print('Core packages preserved:', core_versions_after)
print('GPU:', gpu.name, f'{gpu.total_memory / 2**30:.1f} GiB')
print('Kiểm tra numpy.random:', npr.rand(3))
print('THÀNH CÔNG: môi trường và TagGCM đã sẵn sàng.')


## Bước 4 — Chép ảnh từ Drive vào ổ cục bộ (mỗi runtime mới)

`/content` bị xóa khi runtime ngắt, nên runtime mới phải chép lại. Cell dùng 6 luồng,
có thanh phần trăm và chỉ chép các file còn thiếu. Sau đó nó luôn chạy `split_check.py`.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

DRIVE_ROOT = Path('/content/drive/MyDrive/wild_diff_icmh')
DRIVE_IMAGES = DRIVE_ROOT / 'images' / 'snapshot_kgalagadi'
LOCAL_IMAGES = Path('/content/data/wild_diff_icmh/images')
LOCAL_SITE_IMAGES = LOCAL_IMAGES / 'snapshot_kgalagadi'

assert DRIVE_IMAGES.is_dir(), f'Không thấy dữ liệu: {DRIVE_IMAGES}'
LOCAL_SITE_IMAGES.mkdir(parents=True, exist_ok=True)

print('Đang quét file trên Drive...')
source_files = [path for path in DRIVE_IMAGES.rglob('*') if path.is_file()]
pending = []
for src in source_files:
    dst = LOCAL_SITE_IMAGES / src.relative_to(DRIVE_IMAGES)
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        pending.append((src, dst, src.stat().st_size))

remaining_bytes = sum(size for _, _, size in pending)
print('Tổng file:', len(source_files))
print('Còn phải copy:', len(pending), 'file')
print('Dung lượng còn lại:', f'{remaining_bytes / 2**30:.2f} GiB')

def copy_one(item):
    src, dst, size = item
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.part')
    try:
        with src.open('rb') as fin, tmp.open('wb') as fout:
            while chunk := fin.read(8 * 1024 * 1024):
                fout.write(chunk)
        os.replace(tmp, dst)
        return size
    except Exception:
        if tmp.exists():
            tmp.unlink()
        raise

if pending:
    with ThreadPoolExecutor(max_workers=6) as executor:
        futures = [executor.submit(copy_one, item) for item in pending]
        with tqdm(
            total=remaining_bytes,
            unit='B', unit_scale=True, unit_divisor=1024,
            desc='Copy Kgalagadi',
        ) as progress:
            for future in as_completed(futures):
                progress.update(future.result())
else:
    print('Ảnh cục bộ đã đầy đủ, không cần copy lại.')

subprocess.run([
    sys.executable, 'tools/data/split_check.py',
    'data/manifests/kgalagadi_site_split.jsonl',
    '--build-info', 'data/manifests/build_info.json',
], cwd=REPO, check=True)
print('Ảnh và manifest đã sẵn sàng.')


## Bước 5 — Khôi phục 3 checkpoint từ Drive vào `/content` (mỗi runtime mới)

Checkpoint gốc đã sao lưu trên Drive. Cell có tiến trình, chép song song 2 file và
tự bỏ qua file cục bộ đã đủ. Không cần tải lại Hugging Face.

In [ ]:
DRIVE_CKPT_ROOT = DRIVE_ROOT / 'checkpoints'
CKPT_ROOT = Path('/content/data/wild_diff_icmh/checkpoints')
BPP_WEIGHT = 2
folder = f'CNscale1.0_1_1_{BPP_WEIGHT}_2_WTagGCM_bs16x1_lr0.00005_cfg7.0'

checkpoint_relatives = [
    Path('sd2p1/v2-1_512-ema-pruned.ckpt'),
    Path('ram/ram_plus_swin_large_14m.pth'),
    Path('difficmh_models') / folder / 'model.ckpt',
]
backup_items = []
for relative in checkpoint_relatives:
    src = DRIVE_CKPT_ROOT / relative
    dst = CKPT_ROOT / relative
    assert src.is_file(), f'Thiếu checkpoint trên Drive: {src}'
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        backup_items.append((src, dst, src.stat().st_size))

total_bytes = sum(size for _, _, size in backup_items)
if backup_items:
    def copy_checkpoint(item, progress):
        src, dst, _ = item
        dst.parent.mkdir(parents=True, exist_ok=True)
        tmp = dst.with_name(dst.name + '.part')
        try:
            with src.open('rb') as fin, tmp.open('wb') as fout:
                while chunk := fin.read(8 * 1024 * 1024):
                    fout.write(chunk)
                    progress.update(len(chunk))
            os.replace(tmp, dst)
            return dst
        except Exception:
            if tmp.exists():
                tmp.unlink()
            raise

    with tqdm(
        total=total_bytes,
        unit='B', unit_scale=True, unit_divisor=1024,
        desc='Khôi phục checkpoint',
    ) as progress:
        with ThreadPoolExecutor(max_workers=2) as executor:
            futures = [
                executor.submit(copy_checkpoint, item, progress)
                for item in backup_items
            ]
            for future in as_completed(futures):
                print('\nĐã khôi phục:', future.result())
else:
    print('Checkpoint cục bộ đã đầy đủ, không cần copy lại.')

AUTHOR_CKPT = CKPT_ROOT / 'difficmh_models' / folder / 'model.ckpt'
RAM_CKPT = CKPT_ROOT / 'ram' / 'ram_plus_swin_large_14m.pth'
repo_checkpoints = REPO / 'checkpoints'
if not repo_checkpoints.exists():
    repo_checkpoints.symlink_to(CKPT_ROOT, target_is_directory=True)
elif repo_checkpoints.resolve() != CKPT_ROOT.resolve():
    raise RuntimeError(f'{repo_checkpoints} không trỏ tới {CKPT_ROOT}')

print('RAM++:', RAM_CKPT)
print('Diff-ICMH:', AUTHOR_CKPT)
print('Checkpoint đã sẵn sàng.')


## P2-0 — Đường dẫn chung và checkpoint B0 (λ = 2/8/32)

Checkpoint λ = 8 và 32 chưa có trên Drive thì được tải từ `RuoyuFeng/Diff-ICMH` về Drive một
lần (vài GB), rồi chép sang ổ cục bộ.

In [ ]:
import json
import shutil
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import hf_hub_download

P2 = DRIVE_ROOT / 'phase2'
P2.mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'logs').mkdir(parents=True, exist_ok=True)
MANIFEST = 'data/manifests/kgalagadi_site_split.jsonl'
ILLUMINATION = P2 / 'kgalagadi_illumination.jsonl'
DEV_LIST = P2 / 'kgalagadi_dev.txt'
DETECTIONS = P2 / 'detections' / 'originals.jsonl'
TAGS_ALL = DRIVE_ROOT / 'tags' / 'KGA_all.jsonl'
ARCHIVE = P2 / 'archive'
EVAL_DIR = P2 / 'eval'
RESULTS_REGISTRY = DRIVE_ROOT / 'results' / 'results.jsonl'
SD_CKPT = CKPT_ROOT / 'sd2p1' / 'v2-1_512-ema-pruned.ckpt'
LAMBDAS = [2, 8, 32]
DDIM_STEPS = 50
B0_OVERRIDES = [
    'params.control_stage_config.params.control_model_ratio=1.0',
    'params.c_cfg_scale=3.0',
]
for folder in (ARCHIVE, EVAL_DIR, DETECTIONS.parent):
    folder.mkdir(parents=True, exist_ok=True)


def run_logged(command, log_name, echo=True):
    log_path = DRIVE_ROOT / 'logs' / log_name
    command = [str(part) for part in command]
    print('$', ' '.join(command))
    with log_path.open('a', encoding='utf-8') as log_stream:
        process = subprocess.Popen(
            command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in process.stdout:
            if echo:
                print(line, end='')
            log_stream.write(line)
        return_code = process.wait()
    if return_code:
        raise RuntimeError(f'{log_name}: dừng với mã {return_code}; log đầy đủ: {log_path}')


def author_checkpoint(lam):
    folder = f'CNscale1.0_1_1_{lam}_2_WTagGCM_bs16x1_lr0.00005_cfg7.0'
    relative = Path('difficmh_models') / folder / 'model.ckpt'
    drive_copy = DRIVE_CKPT_ROOT / relative
    if not drive_copy.is_file():
        print(f'Tải checkpoint tác giả λ={lam} từ Hugging Face về Drive...')
        hf_hub_download('RuoyuFeng/Diff-ICMH', relative.as_posix(), local_dir=DRIVE_CKPT_ROOT)
    local = CKPT_ROOT / relative
    if not local.is_file() or local.stat().st_size != drive_copy.stat().st_size:
        local.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(drive_copy, local)
    return local


B0_CKPT = {lam: author_checkpoint(lam) for lam in LAMBDAS}
for lam, path in B0_CKPT.items():
    print(f'B0 λ={lam}: {path} ({path.stat().st_size / 2**30:.2f} GiB)')


### P2-0b — Kiểm tra đủ 10.222 ảnh (tải bù nếu thiếu)

Phase 1 chỉ cần site A01. Nếu Drive chưa có đủ ảnh Kgalagadi, cell tải phần thiếu từ LILA thẳng
vào Drive (chạy tiếp được), rồi chép sang ổ cục bộ. Tải ~9 GB mất khoảng 15–30 phút; nếu thiếu
nhiều, nên chạy riêng cell này trên runtime **CPU** (rẻ hơn GPU nhiều) rồi mới đổi sang L4.

In [ ]:
manifest_rows = [
    json.loads(line) for line in (REPO / MANIFEST).read_text(encoding='utf-8').splitlines() if line.strip()
]
missing = [row for row in manifest_rows if not (LOCAL_IMAGES / row['relative_path']).is_file()]
print(f'Ảnh cục bộ: {len(manifest_rows) - len(missing)}/{len(manifest_rows)}')
if missing:
    run_logged([
        sys.executable, '-u', 'tools/data/download_images.py', '--manifest', MANIFEST,
        '--root', DRIVE_ROOT / 'images', '--workers', 16,
    ], 'p2_download_images.log')
    for row in missing:
        source = DRIVE_ROOT / 'images' / row['relative_path']
        target = LOCAL_IMAGES / row['relative_path']
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source, target)
    still_missing = [row for row in manifest_rows if not (LOCAL_IMAGES / row['relative_path']).is_file()]
    assert not still_missing, f'Vẫn thiếu {len(still_missing)} ảnh, xem logs/p2_download_images.log'
print('Đủ ảnh cho Phase 2.')


## P2-1 — Nhãn ngày/đêm theo nguồn sáng (CPU, vài phút)

Đêm = ảnh do camera tự chiếu sáng (flash hoặc IR), xác định từ chính file ảnh:
EXIF Flash → ảnh xám IR → độ sáng pixel. Không dùng giờ chụp. Kết quả in ra số ảnh theo
từng nguồn tín hiệu và số ảnh mà nhãn theo giờ cũ bị sai.

In [ ]:
run_logged([
    sys.executable, '-u', 'tools/data/label_illumination.py',
    '--manifest', MANIFEST, '--data-root', LOCAL_IMAGES, '--output', ILLUMINATION,
], 'p2_illumination.log')


## P2-2 — Đóng băng tập dev (CPU)

Lần đầu: tạo `phase2/kgalagadi_dev.txt` từ validation, phân tầng theo nhãn ngày/đêm mới.
Các lần sau: tạo lại vào file tạm và **chỉ kiểm tra** là trùng khớp — không bao giờ ghi đè.

In [ ]:
def build_dev(output):
    run_logged([
        sys.executable, '-u', 'tools/data/build_dev_set.py', '--manifest', MANIFEST,
        '--illumination-sidecar', ILLUMINATION, '--output', output,
    ], 'p2_dev_set.log')

def dev_ids(path):
    return [line for line in Path(path).read_text().splitlines() if line and not line.startswith('#')]

if DEV_LIST.is_file():
    check = Path('/content/kgalagadi_dev_check.txt')
    build_dev(check)
    assert dev_ids(check) == dev_ids(DEV_LIST), 'Tập dev tạo lại khác bản đã đóng băng!'
    print('Tập dev đã đóng băng và tái lập đúng:', DEV_LIST)
else:
    build_dev(DEV_LIST)
print(len(dev_ids(DEV_LIST)), 'ảnh dev')


## P2-3 — MegaDetector trên 10.222 ảnh gốc (GPU, env riêng)

MegaDetector V6 cài trong venv riêng `/content/envs/detect` (thư viện ultralytics của nó xung đột
với môi trường codec). Kết quả là pseudo ground truth: một dòng cho **mọi** ảnh, kể cả ảnh không có
con vật. Chạy tiếp được nếu runtime ngắt.

In [ ]:
DETECT_PY = Path('/content/envs/detect/bin/python')
if not DETECT_PY.exists():
    subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', '/content/envs/detect'], check=True)
    subprocess.run([str(DETECT_PY), '-m', 'pip', 'install', '-q', 'PytorchWildlife'], check=True)

run_logged([
    DETECT_PY, '-u', 'tools/detect/run_megadetector.py', '--manifest', MANIFEST,
    '--image-root', LOCAL_IMAGES, '--output', DETECTIONS,
], 'p2_megadetector_originals.log')


## P2-4 — Thống kê miền (CPU)

In [ ]:
run_logged([
    sys.executable, '-u', 'tools/data/domain_stats.py', '--manifest', MANIFEST,
    '--illumination-sidecar', ILLUMINATION, '--detections', DETECTIONS,
    '--output', P2 / 'domain_stats.json',
], 'p2_domain_stats.log')


## P2-5 — RAM++ tags cho cả 20 site (GPU)

Một file cho cả corpus (`tags/KGA_all.jsonl`), dùng cho decode B0 và cho H1 chung ở Phase 3.

In [ ]:
manifest_ids = {
    json.loads(line)['image_id']
    for line in (REPO / MANIFEST).read_text(encoding='utf-8').splitlines() if line.strip()
}
tagged = set()
if TAGS_ALL.is_file():
    for line in TAGS_ALL.read_text(encoding='utf-8').splitlines():
        try:
            tagged.add(json.loads(line)['image_id'])
        except (json.JSONDecodeError, KeyError):
            pass
if manifest_ids <= tagged:
    print(f'Tags đã đủ {len(manifest_ids)} ảnh — bỏ qua.')
else:
    print(f'Còn {len(manifest_ids - tagged)} ảnh chưa có tag.')
    run_logged([
        sys.executable, '-u', 'tools/precompute_ram_tags.py', '--data-root', LOCAL_IMAGES,
        '--checkpoint', RAM_CKPT, '--output', TAGS_ALL,
        '--batch-size', '2', '--num-workers', str(min(2, os.cpu_count() or 1)),
    ], 'p2_ram_tags_all.log')


## P2-6 — Đo thời gian decode B0 và dự báo chi phí P2-7

Decode 3 ảnh dev bằng B0 λ=2 ở cạnh dài 1024 và 512, đọc `decode_log.jsonl`, rồi dự báo giờ GPU
và CU cho P2-7. Chọn `B0_SIDE`, `B0_LIMIT` ở P2-7 theo bảng in ra.

In [ ]:
COLAB_CU_PER_HOUR = 1.54

def b0_command(lam, side, output, limit=None):
    command = [
        sys.executable, '-u', 'inference_partition.py',
        '--ckpt_sd', SD_CKPT, '--ckpt_lc', B0_CKPT[lam],
        '--config', 'configs/model/diffeic.yaml',
        '--input', LOCAL_IMAGES, '--output', output,
        '--manifest', MANIFEST, '--split', 'val', '--dev-list', DEV_LIST,
        '--tag-cache', TAGS_ALL, '--sampler', 'ddim', '--steps', DDIM_STEPS,
        '--device', 'cuda', '--processing-long-side', side, '--skip-existing',
    ]
    if limit:
        command += ['--limit', limit]
    return command + B0_OVERRIDES

n_dev = len(dev_ids(DEV_LIST))
for side in (1024, 512):
    probe = Path('/content/p2_probe') / f'ls{side}'
    run_logged(b0_command(2, side, probe, limit=3), f'p2_probe_ls{side}.log', echo=False)
    records = [json.loads(line) for line in (probe / 'decode_log.jsonl').read_text().splitlines()]
    seconds = sum(r['encode_seconds'] + r['decode_seconds'] for r in records[1:]) / max(1, len(records) - 1)
    hours = seconds * n_dev * len(LAMBDAS) / 3600
    print(f'cạnh dài {side}: {seconds:.1f} s/ảnh (bỏ ảnh đầu khởi động) '
          f'→ {n_dev} ảnh × {len(LAMBDAS)} λ ≈ {hours:.2f} giờ ≈ {hours * COLAB_CU_PER_HOUR:.2f} CU')


## P2-7 — Decode B0 trên tập dev (tốn GPU nhất)

Đọc dự báo ở P2-6, chỉnh `B0_SIDE` / `B0_LIMIT` cho vừa ngân sách, rồi đặt `RUN_B0 = True`.
Kho lưu trữ: `phase2/archive/B0_ls<side>_ddim<steps>/lambda_<λ>/`. Ngắt giữa chừng thì chạy lại
cell — ảnh đã decode được bỏ qua (`--skip-existing`), mỗi ảnh có seed riêng.

In [ ]:
B0_SIDE = 1024
B0_LIMIT = None      # None = cả tập dev; ví dụ 100 nếu dự báo vượt ngân sách
RUN_B0 = False       # đặt True sau khi xem dự báo ở P2-6

if not RUN_B0:
    print('P2-7 chưa chạy: đặt RUN_B0 = True sau khi xem dự báo ở P2-6.')
else:
    for lam in LAMBDAS:
        output = ARCHIVE / f'B0_ls{B0_SIDE}_ddim{DDIM_STEPS}' / f'lambda_{lam}'
        run_logged(b0_command(lam, B0_SIDE, output, limit=B0_LIMIT), f'p2_b0_lambda{lam}.log', echo=False)
        print('Xong B0 λ =', lam, '→', output)


## P2-8 — Baseline JPEG / WebP (CPU, chạy song song 4 tiến trình)

JPEG ở độ phân giải gốc và ở cạnh dài 1024; WebP ở 1024. Cùng giao thức: ảnh gốc vào, ảnh gốc ra.

In [ ]:
CLASSICAL = (
    [('jpeg', q, None) for q in (5, 15, 40)]
    + [('jpeg', q, 1024) for q in (5, 15, 40)]
    + [('webp', q, 1024) for q in (5, 15, 40)]
)

def run_classical(job):
    codec, quality, side = job
    curve = f"{codec}_{'full' if side is None else f'ls{side}'}"
    command = [
        sys.executable, '-u', 'tools/baselines/run_classical.py', '--manifest', MANIFEST,
        '--data-root', LOCAL_IMAGES, '--split', 'val', '--dev-list', DEV_LIST,
        '--codec', codec, '--quality', quality, '--output', ARCHIVE / curve / f'q{quality}',
        '--skip-existing',
    ]
    if side:
        command += ['--processing-long-side', side]
    run_logged(command, f'p2_{curve}_q{quality}.log', echo=False)
    return curve, quality

with ThreadPoolExecutor(max_workers=4) as pool:
    for curve, quality in pool.map(run_classical, CLASSICAL):
        print('Xong', curve, 'q', quality)


## P2-9 — Baseline CompressAI (GPU, nhẹ)

`bmshj2018-hyperprior` (họ codec của bài Xie 2025), `mbt2018`, `cheng2020-attn`, chất lượng 1–3,
ở cạnh dài 1024; entropy coding thật, bitstream ghi ra file.

In [ ]:
for model in ('bmshj2018-hyperprior', 'mbt2018', 'cheng2020-attn'):
    for quality in (1, 2, 3):
        curve = f'compressai-{model}_ls1024'
        run_logged([
            sys.executable, '-u', 'tools/baselines/run_compressai_zoo.py', '--manifest', MANIFEST,
            '--data-root', LOCAL_IMAGES, '--split', 'val', '--dev-list', DEV_LIST,
            '--model', model, '--quality', quality, '--processing-long-side', 1024,
            '--output', ARCHIVE / curve / f'q{quality}', '--skip-existing',
        ], f'p2_{curve}_q{quality}.log', echo=False)
        print('Xong', curve, 'q', quality)


## P2-10 — Chấm điểm mọi kho lưu trữ

Với mỗi `phase2/archive/<đường>/<điểm>/`: chỉ số ảnh (PSNR, SSIM ×2, MS-SSIM, LPIPS, DISTS, bpp,
compression ratio, thời gian) + MegaDetector trên ảnh tái tạo → mAP, ảnh rỗng báo nhầm, ảo giác,
mất con vật. Tất cả ghi vào `results/results.jsonl` với `exp_id = p2dev_<đường>__<điểm>`.
Kho đã chấm thì bỏ qua.

In [ ]:
try:
    import pycocotools  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pycocotools'], check=True)

for run_info in sorted(ARCHIVE.glob('*/*/run_info.json')):
    root = run_info.parent
    curve, point = root.parent.name, root.name
    name = f'{curve}__{point}'
    image_eval = EVAL_DIR / f'{name}.jsonl'
    machine_eval = EVAL_DIR / f'{name}.machine.json'
    if machine_eval.is_file():
        print('Đã chấm:', name)
        continue
    info = json.loads(run_info.read_text())
    common = ['--manifest', MANIFEST, '--split', 'val', '--dev-list', DEV_LIST,
              '--illumination-sidecar', ILLUMINATION]
    run_logged([
        sys.executable, '-u', 'tools/evaluate_kgalagadi.py', *common,
        '--data-root', LOCAL_IMAGES, '--reconstruction-root', root,
        '--detections', DETECTIONS, '--method', curve, '--lpips', '--dists',
        '--output', image_eval, '--results-registry', RESULTS_REGISTRY,
        '--exp-id', f'p2dev_{name}', '--lambda-rate', point,
        '--ddim-steps', info.get('steps', 0),
    ], f'p2_eval_{name}.log', echo=False)
    predictions = DETECTIONS.parent / f'{name}.jsonl'
    run_logged([
        DETECT_PY, '-u', 'tools/detect/run_megadetector.py', '--manifest', MANIFEST,
        '--split', 'val', '--dev-list', DEV_LIST, '--image-root', root, '--suffix', '.png',
        '--output', predictions,
    ], f'p2_megadetector_{name}.log', echo=False)
    run_logged([
        sys.executable, '-u', 'tools/eval_machine.py', *common,
        '--gt', DETECTIONS, '--pred', predictions, '--output', machine_eval,
        '--results-registry', RESULTS_REGISTRY, '--exp-id', f'p2dev_{name}',
        '--method', curve, '--lambda-rate', point, '--ddim-steps', info.get('steps', 0),
    ], f'p2_machine_{name}.log', echo=False)
    print('Đã chấm:', name)


## P2-11 — Đồ thị RD chung (tiêu chí 7 của Phase 2)

Mỗi đường là một phương pháp; trục x là bpp tính trên pixel ảnh gốc (log). Dùng để thấy dải bitrate
chồng lấn giữa B0 và các baseline trước khi chọn λ cho H1.

In [ ]:
import matplotlib.pyplot as plt

points = {}
for line in RESULTS_REGISTRY.read_text(encoding='utf-8').splitlines():
    row = json.loads(line)
    if not str(row['exp_id']).startswith('p2dev_'):
        continue
    if row['illumination'] != 'all' or row.get('subset', 'all') != 'all':
        continue
    points.setdefault(row['exp_id'], {})[row['metric']] = row['value']

curves = {}
for exp_id, values in points.items():
    curve = exp_id[len('p2dev_'):].split('__')[0]
    curves.setdefault(curve, []).append(values)

panels = ['psnr', 'ms_ssim', 'lpips', 'dists', 'map', 'hallucination_rate']
figure, axes = plt.subplots(2, 3, figsize=(16, 9))
for axis, metric in zip(axes.flat, panels):
    for curve, values in sorted(curves.items()):
        series = sorted((v['bpp'], v[metric]) for v in values if 'bpp' in v and v.get(metric) is not None)
        if series:
            axis.plot(*zip(*series), marker='o', label=curve)
    axis.set_xscale('log')
    axis.set_xlabel('bpp (pixel ảnh gốc)')
    axis.set_title(metric)
    axis.grid(alpha=0.3)
axes.flat[0].legend(fontsize=8)
figure.tight_layout()
figure.savefig(P2 / 'rd_dev.png', dpi=120)
plt.show()
print('Đồ thị:', P2 / 'rd_dev.png')


## P2-12 — Ghi CU đo thật của phiên

Mở **Runtime → View resources**, chép số "Available" vào `CU_AVAILABLE_NOW`, chạy cell, rồi ngắt runtime.

In [ ]:
from datetime import datetime, timezone

CU_AVAILABLE_NOW = None  # số "Available" ngay lúc này

if CU_AVAILABLE_AT_START is None or CU_AVAILABLE_NOW is None:
    raise ValueError('Điền CU_AVAILABLE_AT_START (Bước 1) và CU_AVAILABLE_NOW rồi chạy lại cell.')
session = {
    'date': datetime.now(timezone.utc).isoformat(),
    'gpu': torch.cuda.get_device_name(0),
    'cu_start': float(CU_AVAILABLE_AT_START),
    'cu_end': float(CU_AVAILABLE_NOW),
    'cu_consumed': float(CU_AVAILABLE_AT_START) - float(CU_AVAILABLE_NOW),
    'git_commit': subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO, text=True).strip(),
}
with (P2 / 'sessions.jsonl').open('a', encoding='utf-8') as stream:
    stream.write(json.dumps(session) + chr(10))
print(session)
print('Ngắt runtime ngay (Runtime → Disconnect and delete runtime) để ngừng tiêu CU.')
